# Apply governance

Runs at the end of the job and stamps Unity Catalog metadata onto every table and
view from `governance/config/tables.json`:

* table and column **comments** — the data dictionary that Catalog Explorer, Power BI and Genie read
* **tags** (layer, domain, classification) and **PII column tags**, which is what makes masking and audits possible later
* informational **primary and foreign keys** — Power BI builds relationships from these, and Genie uses them to write joins
* schema-level **grants**

Separate from the pipeline on purpose: this is metadata, it is idempotent, and it
has no business being interleaved with the code that moves rows. Every statement is
best-effort — a workspace that rejects tags on views must not fail the job.

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.governance import (
    grant_on_schema,
    qualified,
    set_column_comments,
    set_foreign_key,
    set_owner,
    set_primary_key,
    set_table_comment,
    set_tags,
    tag_columns,
)
from common_utils.logger import get_logger, log_info, log_warning
from common_utils.settings import load_json

In [ ]:
dbutils.widgets.text("config_path", "governance/config/tables.json")
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")

catalog = dbutils.widgets.get("catalog")
schema_names = {
    "bronze": dbutils.widgets.get("bronze_schema"),
    "silver": dbutils.widgets.get("silver_schema"),
    "gold": dbutils.widgets.get("gold_schema"),
}
config = load_json(dbutils.widgets.get("config_path"))
audit_comments = config.get("audit_column_comments", {})
default_tags = config.get("default_tags", {})
owner = config.get("owner")
logger = get_logger("governance")

In [ ]:
applied, skipped = 0, 0

for table in config["tables"]:
    schema = schema_names[table["schema"]]
    name = table["name"]
    full_name = f"{catalog}.{schema}.{name}"
    is_view = bool(table.get("is_view"))

    if not spark.catalog.tableExists(full_name):
        log_warning(logger, "object missing, skipped", object=full_name)
        skipped += 1
        continue

    target = qualified(catalog, schema, name)
    existing = {field.name for field in spark.table(full_name).schema.fields}

    # Comments: the table's own, plus the shared audit-column wording, for whichever
    # audit columns this object actually has.
    set_table_comment(spark, target, table.get("comment", ""))
    comments = {**{c: t for c, t in audit_comments.items() if c in existing}, **table.get("column_comments", {})}
    set_column_comments(spark, target, {c: t for c, t in comments.items() if c in existing}, is_view=is_view)

    set_tags(spark, target, {**default_tags, **table.get("tags", {})}, is_view=is_view)
    tag_columns(spark, target, [c for c in table.get("pii_columns", []) if c in existing], is_view=is_view)

    if not is_view:
        set_primary_key(spark, target, table.get("primary_key", []), name=f"pk_{name}")
        for fk in table.get("foreign_keys", []):
            set_foreign_key(
                spark,
                target,
                name=f"fk_{name}_{fk['references']}",
                columns=fk["columns"],
                references=qualified(catalog, schema, fk["references"]),
                referenced_columns=fk["referenced_columns"],
            )

    set_owner(spark, target, owner, is_view=is_view)
    applied += 1
    log_info(logger, "governed", object=full_name, is_view=is_view)

## Grants
Least privilege: consumers get `SELECT` on Gold, nothing else. Empty in dev because
the groups do not exist yet — fill in `grants` in the config for production.

In [ ]:
for layer, schema in schema_names.items():
    grants = config.get("grants", {}).get(layer, {})
    grant_on_schema(spark, catalog, schema, grants)

print(f"governed {applied} objects, skipped {skipped}")

## Check what landed
Everything a BI tool or an AI assistant reads about your model comes from here.

In [ ]:
display(
    spark.sql(
        f"""
        SELECT table_schema, table_name, column_name, comment
        FROM {catalog}.information_schema.columns
        WHERE table_schema = '{schema_names["gold"]}' AND comment IS NOT NULL
        ORDER BY table_name, ordinal_position
        """
    )
)